# MeloLegacy AI — Staff-line segmentation baseline

**Milestone 0:** train a small segmentation model to locate the five staff lines in score images. This is a reproducible bootstrap, not a complete OMR model. Synthetic training data is used only to verify the training pipeline; real-score validation is mandatory before production use.

Run in Google Colab with Runtime → Change runtime type → T4 GPU (if available).

In [ ]:
import os, random, math, time
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
print('PyTorch:', torch.__version__)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)


## 1. Generate controlled synthetic staff images

The target mask marks staff-line pixels. Random spacing, vertical offset, slight skew, blur/noise and note-like distractors create variation. These examples validate the code path only; they do not replace licensed real score images.

In [ ]:
try:
    import cv2
except ImportError:
    !pip -q install opencv-python-headless
    import cv2

H, W = 128, 384

def make_sample(seed=None):
    rng = np.random.default_rng(seed)
    img = np.full((H, W), 245, np.uint8)
    mask = np.zeros((H, W), np.uint8)
    top = int(rng.integers(22, 55)); gap = int(rng.integers(7, 13))
    slope = float(rng.uniform(-0.025, 0.025))
    thickness = int(rng.integers(1, 3))
    for k in range(5):
        y0 = top + k*gap
        pts = np.array([[0, int(y0)], [W-1, int(y0+slope*(W-1))]], np.int32)
        cv2.line(img, tuple(pts[0]), tuple(pts[1]), 35, thickness, cv2.LINE_AA)
        cv2.line(mask, tuple(pts[0]), tuple(pts[1]), 255, thickness, cv2.LINE_AA)
    # Distractor noteheads, stems and speckle; not part of the target mask.
    for _ in range(int(rng.integers(5, 18))):
        x=int(rng.integers(8,W-8)); y=int(rng.integers(12,H-12))
        cv2.ellipse(img,(x,y),(int(rng.integers(3,6)),2),-20,0,360,45,-1)
        if rng.random()<0.7: cv2.line(img,(x+4,y-15),(x+4,y+1),50,1)
    noise=rng.normal(0, float(rng.uniform(1,9)), (H,W))
    img=np.clip(img.astype(np.float32)+noise,0,255).astype(np.uint8)
    if rng.random()<0.3: img=cv2.GaussianBlur(img,(3,3),0.5)
    return img, mask

class StaffDataset(Dataset):
    def __init__(self,n,offset=0): self.n=n; self.offset=offset
    def __len__(self): return self.n
    def __getitem__(self,i):
        x,y=make_sample(self.offset+i)
        return torch.from_numpy(x[None].astype('float32')/255), torch.from_numpy((y[None]>0).astype('float32'))

train_ds=StaffDataset(1200,0); val_ds=StaffDataset(200,100000)
train_dl=DataLoader(train_ds,batch_size=24,shuffle=True,num_workers=0)
val_dl=DataLoader(val_ds,batch_size=24,shuffle=False,num_workers=0)
x,y=make_sample(7)
plt.figure(figsize=(12,3)); plt.subplot(1,2,1); plt.imshow(x,cmap='gray'); plt.title('Synthetic score'); plt.axis('off'); plt.subplot(1,2,2); plt.imshow(y,cmap='gray'); plt.title('Staff-line target'); plt.axis('off'); plt.show()


## 2. Define a compact segmentation network

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self,a,b):
        super().__init__(); self.net=nn.Sequential(nn.Conv2d(a,b,3,padding=1),nn.BatchNorm2d(b),nn.ReLU(inplace=True),nn.Conv2d(b,b,3,padding=1),nn.BatchNorm2d(b),nn.ReLU(inplace=True))
    def forward(self,x): return self.net(x)

class StaffNet(nn.Module):
    def __init__(self):
        super().__init__(); self.e1=ConvBlock(1,16); self.e2=ConvBlock(16,32); self.mid=ConvBlock(32,64); self.pool=nn.MaxPool2d(2); self.up2=nn.ConvTranspose2d(64,32,2,2); self.d2=ConvBlock(64,32); self.up1=nn.ConvTranspose2d(32,16,2,2); self.d1=ConvBlock(32,16); self.out=nn.Conv2d(16,1,1)
    def forward(self,x):
        a=self.e1(x); b=self.e2(self.pool(a)); c=self.mid(self.pool(b)); z=self.d2(torch.cat([self.up2(c),b],1)); z=self.d1(torch.cat([self.up1(z),a],1)); return self.out(z)

model=StaffNet().to(device)
optimizer=torch.optim.AdamW(model.parameters(),lr=1e-3)
criterion=nn.BCEWithLogitsLoss()
print('Parameters:',sum(p.numel() for p in model.parameters()))


## 3. Train and evaluate

The validation set uses different random seeds. Pixel precision/recall/F1 are reported for the synthetic task only.

In [ ]:
def run_epoch(loader,training):
    model.train(training); total=0; loss_sum=0; tp=fp=fn=0
    for xb,yb in loader:
        xb,yb=xb.to(device),yb.to(device)
        with torch.set_grad_enabled(training):
            logits=model(xb); loss=criterion(logits,yb)
            if training:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
        total+=len(xb); loss_sum+=loss.item()*len(xb)
        pred=(torch.sigmoid(logits)>0.5); truth=(yb>0.5)
        tp+=int((pred&truth).sum()); fp+=int((pred&~truth).sum()); fn+=int((~pred&truth).sum())
    precision=tp/(tp+fp+1e-8); recall=tp/(tp+fn+1e-8); f1=2*precision*recall/(precision+recall+1e-8)
    return loss_sum/total,precision,recall,f1

history=[]; epochs=8
for epoch in range(1,epochs+1):
    tr=run_epoch(train_dl,True); va=run_epoch(val_dl,False)
    history.append((tr[0],va[0],va[3]))
    print(f'Epoch {epoch:02d}/{epochs} | train_loss={tr[0]:.4f} | val_loss={va[0]:.4f} | synthetic_val_P={va[1]:.3f} R={va[2]:.3f} F1={va[3]:.3f}')

plt.plot([h[0] for h in history],label='train loss'); plt.plot([h[1] for h in history],label='validation loss'); plt.legend(); plt.title('Training curves'); plt.show()


## 4. Save model weights and test one image

Do not use this checkpoint as a production OMR model until it has been tested on real printed PDFs and scans with ground-truth line coordinates.

In [ ]:
os.makedirs('/content/melolegacy-artifacts',exist_ok=True)
checkpoint='/content/melolegacy-artifacts/staffline-segmentation-baseline.pt'
torch.save({'model_state_dict':model.cpu().state_dict(),'model':'StaffNet','input_size':[H,W],'task':'synthetic staff-line segmentation','training_samples':len(train_ds),'validation_samples':len(val_ds)},checkpoint)
print('Saved:',checkpoint)

model=model.to(device).eval()
test_img,_=make_sample(2026)
with torch.no_grad(): prob=torch.sigmoid(model(torch.from_numpy(test_img[None,None].astype('float32')/255).to(device)))[0,0].cpu().numpy()
plt.figure(figsize=(13,3)); plt.subplot(1,3,1); plt.imshow(test_img,cmap='gray'); plt.title('Input'); plt.axis('off'); plt.subplot(1,3,2); plt.imshow(prob,cmap='magma',vmin=0,vmax=1); plt.title('Predicted line probability'); plt.axis('off'); plt.subplot(1,3,3); plt.imshow(prob>0.5,cmap='gray'); plt.title('Predicted mask'); plt.axis('off'); plt.show()

from google.colab import files
files.download(checkpoint)


## 5. Next real-data phase — required before deployment

1. Obtain MUSCIMA++ v2 annotations and its corresponding CVC-MUSCIMA page images. The dataset is **CC BY-NC-SA 4.0** and is non-commercial/share-alike; preserve both required dataset attributions. See the official sources below.
2. Parse MuNG annotations and map every annotated page to its image. Split train/validation/test by page/writer, never random crops from the same page across splits.
3. Train symbol detection/classification (noteheads, stems, beams, rests, clefs, accidentals) and graph relations separately from staff-line segmentation.
4. Evaluate on real held-out pages; report per-class precision/recall, staff-line coordinate error, symbol AP and notation-level error. Keep uncertain results flagged.
5. Only after validation, add pitch/rhythm interpretation and MusicXML export.

Official dataset: https://github.com/OMR-Research/muscima-pp

Dataset documentation and download: https://ufal.mff.cuni.cz/muscima

Image source / terms: https://pages.cvc.uab.es/cvcmuscima/index_database.html
